# Concept-Direction Latent Dynamics

Where along the forward pass does a concept direction live? This notebook builds the color-vs-fruit concept direction
of the [concept-direction steering notebook](ct_concept_steering_demo.ipynb) (the `orange` sense-disambiguation
example) from each state `gemma-3-1b-it` computes on the way to its answer, and compares those directions stage by
stage:

- **Embedding stages** build the direction from token embeddings: the two target tokens (`Fruit`, `Color`), the mean
  embeddings of the concept words, and their paired rejection.
- **Answer-state stages** build it from the residual stream at the answer position of a set of probe prompts, one
  group per sense.
- **Context-state stages** build it from the residual stream at the concept word in context, its projection onto the
  answer states, and their paired residuals.

Each stage's direction is then examined four ways: how the directions align with each other, what each one does when
added after the final norm (a readout of its alignment with the unembedding), what each does when added mid-model
(causal steering), and which transcoder features its attribution graph selects. A final section projects the
per-example states and the stage directions into two dimensions.

The analysis itself lives in the `speediedan/concept-direction` Hub component
(`exp/analysis/concept_direction_latent_dynamics.py`), fetched at a pinned revision. This notebook runs it once and
lays out its report.

**Requirements:** a bf16 CUDA GPU (about 4.5 GB peak), the gated `google/gemma-3-1b-it` weights, the `examples` extra
(`plotly`, `umap-learn`) and the `git-deps` group (circuit-tracer).

In [ ]:
# Parameters - These will be injected by papermill during parameterized test runs
# -- The experiment component ---------------------------------------------------------------------
# Pinned, not floating: the revision you read is the revision that executes.
COMPONENT_REPO = "speediedan/concept-direction"
COMPONENT_REVISION = "1b173742869f717caf218f2e6afb4deb8f94d0e5"
EXPERIMENT_KEY = "gemma3_1b_it_local_latent_dynamics_color_fruit_orange_fs_l10_n5"
# False resolves the revision from the local components cache only: offline, after an earlier fetch.
FETCH_COMPONENT = True
# The analysis is Python that component publishes, run in this kernel. Left False, the fetch cell stops with the
# reason and where the fetched code sits, rather than opting in on your behalf.
TRUST_COMPONENT_CODE = False
# Rows shown in the top-features table for each stage.
TOP_FEATURES_SHOWN = 5
# With a reachable local Neuronpedia stack, feature explanations missing there are generated (this needs the
# explanation CLI's credentials, see the local-Neuronpedia steering demo); without one, the feature table reads
# public Neuronpedia and says so.
GENERATE_MISSING_LOCAL_EXPLANATIONS = True

## 1. Fetch the experiment, then run it

`it.hub.pull_experiment` fetches one experiment definition and the files it owns, pinned to one commit, and returns
the snapshot directory those files sit in (`it.hub.load_experiment` resolves the same from the local cache, without
the network). Nothing executes until the trust decision below; the analysis then builds
its own session from the experiment configuration (model, transcoders, prompts, stages), so this cell is the whole
run. Expect a few minutes on a single GPU: every stage builds a direction, sweeps two steering sites and computes an
attribution graph. Its log is long, so the rendered page collapses it; the sections below lay out the report.

In [ ]:
# @title 1: Fetch and run { display-mode: "form" }
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

import interpretune as it

for _env_candidate in (Path.cwd() / ".env", Path.home() / "repos" / "interpretune" / ".env"):
    if _env_candidate.exists():
        load_dotenv(_env_candidate)
        break

if FETCH_COMPONENT:
    canonical_key, _, snapshot_dir = it.hub.pull_experiment(COMPONENT_REPO, EXPERIMENT_KEY, revision=COMPONENT_REVISION)
else:
    canonical_key, _, snapshot_dir = it.hub.load_experiment(COMPONENT_REPO, EXPERIMENT_KEY, revision=COMPONENT_REVISION)
print(f"fetched {COMPONENT_REPO}:{canonical_key} into {snapshot_dir.name}")

if not TRUST_COMPONENT_CODE:
    raise SystemExit(
        "Set TRUST_COMPONENT_CODE = True to continue.\n"
        f"The experiment is already cached at {snapshot_dir}, so you can read "
        "exp/analysis/concept_direction_latent_dynamics.py before deciding."
    )
os.environ["IT_TRUST_REMOTE_CODE"] = "1"
sys.path.insert(0, str(snapshot_dir))

from exp.analysis.concept_direction_latent_dynamics import run_latent_dynamics_analysis  # noqa: E402

# The fetch checks that the configuration's filename matches its key, so the key names exactly one file.
config_paths = sorted(snapshot_dir.rglob(f"{canonical_key}.yaml"))
assert len(config_paths) == 1, f"expected one {canonical_key}.yaml in the snapshot, found {config_paths}"
report = run_latent_dynamics_analysis(config_paths[0])
print(f"{len(report['stages'])} stages, {len(report['examples'])} probe examples")
print(f"store-state reconstruction check, largest difference: {report['reconstruction_max_abs_diff']:.3g}")

## 2. The prompts, the states, the stages and the probe examples

The concept direction is built from probe prompts: one classification question, asked of each word in two groups (the
fruits and the colors), each with its expected answer. The analysis prompt the direction is then steered on is the
`orange` question, which the model answers `Color`.

Each probe prompt is run with its expected answer appended, and four **states** are read from the residual stream
(after the final norm, at `unembed.hook_in`):

- the **answer state**, at the expected answer's token;
- the **context state**, at the probe word's last token;
- the **projected state**, the answer state's component along the context state (the context-enhanced extraction);
- the **final state**, what the store keeps: the projected state here, since every probe word is found in its prompt.

Every stage is one concept direction: its **basis** (token embeddings, or one of those states), its **mode** (a plain
mean difference between the two groups, or a paired rejection, below), and, for the store stages, the **raw norm** of
the group separation before it is normalized. The embedding stages read no prompt at all: they take rows of the
embedding matrix for the group words and the two target words.

The example table is the sanity check behind everything else: each kept probe example's word, its expected answer, the
model's margin for it (the expected answer's logit minus the other target's), and the weight it carries. The store keeps
only the examples the model answers as expected (`store_concept_correct_only: true` in this configuration) and weights
each by its margin; an example answered the other way would otherwise pull the example-weighted stages toward the
wrong sense, with a weight as large as its error. The `unweighted` stage counts every kept example equally.

<details>
<summary><b><code>paired_rejection</code> (a direction-building mode) and <code>reject</code> (an intervention op): how they relate, and when to use each</b></summary>

Both remove a vector's component along another, $x - \frac{x^\top y}{y^\top y} y$, and they are used at different times
for different jobs.

- **`paired_rejection`** is a mode of `it.concept_direction`, used once, to *build* a direction. For each pair (the
  $i$-th fruit with the $i$-th color) it removes the color row's direction from the fruit row, then averages what is left
  and normalizes. What the pair shares (that both are short common nouns in the same question) cancels, so the direction
  keeps what separates the senses. Use it when the two groups can be paired and share a large common component, as two
  sets of words in the same template do; use `mean_difference` when they cannot be paired.
- **`reject`** is an intervention op, applied *during a forward pass* to an activation: it removes the activation's
  component along a given direction at a hook site, so the model computes on without it. Use it to ask whether the model
  needs that direction there: an ablation along one axis, leaving everything orthogonal to it untouched.

A direction built by `paired_rejection` can be handed to `reject` (remove the concept), or to `add` (steer along it),
which is what this notebook's steering sections do.
</details>

In [ ]:
# @title 2: Prompts, stage and example tables { display-mode: "form" }
from exp.analysis.concept_direction_analysis import build_classification_prompt_text
from exp.concept_direction import load_concept_pair
from interpretune.harness.config import load_experiment_config

from it_examples.utils.nb_ui_utils import (
    NEGATIVE_COLOUR,
    POSITIVE_COLOUR,
    SIGN_LEGEND,
    Signed,
    display_notebook_table,
    display_plotly_figure,
)

payload = load_experiment_config(config_paths[0])
# the concept pair is a sibling of the experiment configuration in the snapshot
pair = load_concept_pair(config_paths[0].parent / payload["EXPERIMENT"]["concept_pair_config_path"])
examples = report["examples"]
# the store keeps only the prompts answered as expected, so each kept example names the prompt it came from
example_of_prompt = {ex["prompt_index"]: ex["example_index"] for ex in examples}
probe_prompts = [
    (group, word, answer)
    for group, entities in ((pair.group_a_name, pair.group_a_entities), (pair.group_b_name, pair.group_b_entities))
    for word, answer in entities
]
display_notebook_table(
    [
        (
            index,
            group,
            word,
            answer,
            build_classification_prompt_text(word, pair.classification_question),
            example_of_prompt.get(index, "not kept: answered as the other target"),
        )
        for index, (group, word, answer) in enumerate(probe_prompts)
    ],
    ["prompt", "group", "word", "expected answer", "probe prompt", "store example"],
    title=f"The probe prompts ({pair.concept_label})",
    caption=f"Each is rendered with the chat template. The analysis prompt: {payload['PROMPT']['text']!r}",
    emphasize=[index for index in range(len(probe_prompts)) if index not in example_of_prompt],
)

display_notebook_table(
    [
        (name, rec["basis"], rec["mode"], "" if rec.get("raw_norm") is None else f"{rec['raw_norm']:.1f}")
        for name, rec in report["stages"].items()
    ],
    ["stage", "basis", "mode", "raw norm"],
    title="The stages",
    caption="Embedding stages have no store separation, so no raw norm. Store-state reconstruction check, largest "
    f"difference: {report['reconstruction_max_abs_diff']:.3g}.",
)

wrong = [ex for ex in examples if not ex["answered_as_expected"]]
display_notebook_table(
    [
        (
            ex["example_index"],
            ex["prompt_index"],
            ex["word"],
            ex["group"],
            ex["expected_answer"],
            Signed(ex["expected_answer_margin"]),
            "yes" if ex["answered_as_expected"] else "no",
            f"{ex['weight']:.2f}",
            f"{ex['answer_context_cosine']:.3f}",
        )
        for ex in examples
    ],
    [
        "example",
        "prompt",
        "word",
        "group",
        "expected answer",
        "margin",
        "answered as expected",
        "weight",
        "answer/context cosine",
    ],
    title=f"{len(examples) - len(wrong)} of {len(examples)} probe examples answered as expected",
    caption="The store's examples: the margin is the expected answer's logit minus the other target's.",
    legend=SIGN_LEGEND,
    emphasize=[i for i, ex in enumerate(examples) if not ex["answered_as_expected"]],
)

### Where each state is read

The token strips below mark, for one probe example of each group, the positions the store states come from, and, for
the analysis prompt, the position each stage's attribution graph targets (the last position, where the model writes its
answer). Hover a token for its position. The embedding stages read none of these positions.

In [ ]:
# @title 2b: Token positions { display-mode: "form" }
from transformers import AutoTokenizer

from interpretune.harness.sessions import resolve_model_spec
from it_examples.utils.nb_ui_utils import display_token_positions

spec = resolve_model_spec(payload["MODEL"]["family"], payload["MODEL"]["variant"])
tokenizer = AutoTokenizer.from_pretrained(spec.model_name)
COLOURS = {"probe word": "#8250df", "context state": "#1a7f37", "answer state": "#bc4c00", "graph target": "#bc4c00"}

for group in (pair.group_a_name, pair.group_b_name):
    ex = next(ex for ex in examples if ex["group"] == group)
    marks = {pos: ("probe word", COLOURS["probe word"]) for pos in range(ex["probe_span"][0], ex["probe_span"][1] + 1)}
    marks[ex["context_index"]] = ("context state", COLOURS["context state"])  # the word's last token
    marks[ex["answer_index"]] = ("answer state", COLOURS["answer state"])
    display_token_positions(
        ex["tokens"],
        marks,
        title=f"Store example {ex['example_index']} ({ex['word']!r}, {group}): the prompt with its expected answer",
        caption="The projected and final states combine the answer and context states, so they have no position of "
        "their own.",
    )

analysis_ids = next(iter((report.get("stage_graphs") or {}).values()))["rendered_prompt_token_ids"]
analysis_tokens = tokenizer.convert_ids_to_tokens(analysis_ids)
analysis_marks = {len(analysis_tokens) - 1: ("graph target", COLOURS["graph target"])}
display_token_positions(
    analysis_tokens,
    analysis_marks,
    title="The analysis prompt: each stage's attribution graph targets the last position",
    caption="The steering sections add each direction at their site; the feature table's positions index this prompt.",
)

## 3. How the stage directions align

Cosine similarity between every pair of stage directions, all in the residual stream's coordinates. Blocks along the
diagonal are stages that agree on the concept; near-zero entries between blocks mean two stages read the same concept
along directions that have nothing in common.

In [ ]:
# @title 3: Stage cosine matrix { display-mode: "form" }
import plotly.express as px

cosine_stages = report["cosine"]["stages"]
fig = px.imshow(
    report["cosine"]["matrix"].float().numpy(),
    x=cosine_stages,
    y=cosine_stages,
    zmin=-1,
    zmax=1,
    color_continuous_scale="RdBu_r",
    text_auto=".2f",
    title="Cosine similarity between stage directions",
)
fig.update_layout(height=650, width=850)
display_plotly_figure(fig, "it-stage-cosine")

## 4. Steering: a readout after the final norm, and a causal edit at block 10

The target gap is $\mathrm{logit}(\text{Fruit}) - \mathrm{logit}(\text{Color})$; the prompt asks about `orange` and
the model answers `Color`, so it starts negative.

**At `unembed.hook_in`** each unit direction is added at fixed scales after the final norm. Nothing downstream can
transform it there, so the gap change is exactly linear in scale: this column measures how well each direction aligns
with the difference of the two target unembeddings, not what it does to the computation.

**At block 10** each stage is recomputed from the states at that hook and added there, at fractions of the stage's own
group separation at that site (residual norms there are hundreds, so fixed absolute scales would move little). This is
the causal column: the edit passes through every later layer.

In [ ]:
# @title 4: Steering tables { display-mode: "form" }
# the gap is logit(Fruit) - logit(Color): a rise moves the answer toward Fruit
GAP_LEGEND = ((POSITIVE_COLOUR, "gap rises (toward Fruit)"), (NEGATIVE_COLOUR, "gap falls (toward Color)"))


def display_sweep(per_stage, key, label, *, title, caption):
    """One row per stage (in report order), one column per swept value, each cell the signed gap change."""
    values = [rec[key] for rec in next(iter(per_stage.values()))]
    display_notebook_table(
        [(name, *(Signed(rec["gap_delta"]) for rec in recs)) for name, recs in per_stage.items()],
        ["stage", *(label(v) for v in values)],
        title=title,
        caption=caption,
        legend=GAP_LEGEND,
    )


display_sweep(
    report["steering"],
    "scale_factor",
    lambda v: f"scale {v:g}",
    title=f"Readout at {report['steering_site']}: gap change per scale",
    caption="Each stage's unit direction added after the final norm: linear in scale, so a readout of alignment.",
)
causal = report["causal_steering"]
display_sweep(
    causal["efficacy"],
    "fraction",
    lambda v: f"{v:g}x separation",
    title=f"Causal steering at {causal['hook']}: gap change per fraction of each stage's separation",
    caption="Each stage recomputed from the states at this hook and added there; every later layer transforms it.",
)

## 5. Which features each stage's attribution graph selects

For each stage, one attribution graph with the stage's direction as its target, its top features (layers 10 and up,
ranked by absolute signed influence), and the gap change when those features are intervened on. A feature is a
transcoder latent `(layer, position, index)`: its **encoder** vector reads the residual stream at the transcoder's input
hook of that layer, and its **decoder** vector writes back at the output hook, which is where the intervention acts.
The position indexes the chat-rendered prompt, decoded beside it. Explanations come from the Neuronpedia instance the
experiment configures, where one is available for the feature.

In [ ]:
# @title 5: Per-stage top features { display-mode: "form" }
import yaml
from huggingface_hub import hf_hub_download

from interpretune.harness.notebook_ui import resolve_feature_explanations
from interpretune.utils.neuronpedia_db_utils import check_local_neuronpedia_services
from interpretune.utils.neuronpedia_explanations import (
    ensure_local_feature_explanations,
    feature_tuples_to_feature_refs,
)

graphs = report.get("stage_graphs") or {}
assert graphs, "the experiment ran without stage_graphs: enable LATENT_DYNAMICS.stage_graphs to fill this section"
prompt_ids = next(iter(graphs.values()))["rendered_prompt_token_ids"]

# the transcoder set names the hooks its encoders read and its decoders write
tc_repo_owner, tc_repo_name, tc_subfolder = spec.transcoder_set.split("/", 2)
with open(hf_hub_download(f"{tc_repo_owner}/{tc_repo_name}", "config.yaml", subfolder=tc_subfolder)) as f:
    tc_hooks = yaml.safe_load(f)

# The configuration asks for a local Neuronpedia; use it only if its webapp and database both answer, and say so if not.
np_cfg = payload.get("NEURONPEDIA") or {}
services = check_local_neuronpedia_services(
    local_db_url=np_cfg.get("local_db_url"), webapp_url=np_cfg["local_webapp_url"]
)
local_np = bool(np_cfg.get("use_localhost")) and services.webapp_available and services.db_available
np_base_url = np_cfg["local_webapp_url"] if local_np else "https://www.neuronpedia.org"
# features as plain ints, so the explanation lookup keys match
shown = {
    name: [(tuple(int(v) for v in fid), float(score)) for fid, score in zip(g["feature_ids"], g["feature_scores"])][
        :TOP_FEATURES_SHOWN
    ]
    for name, g in graphs.items()
}
shown_pairs = {(layer, index) for feats in shown.values() for (layer, _, index), _ in feats}
explanation_note = "read from public Neuronpedia, since no local Neuronpedia webapp and database answered"
if local_np:
    coverage = ensure_local_feature_explanations(
        feature_tuples_to_feature_refs(
            model_id=spec.neuronpedia_model,
            source_set=spec.neuronpedia_set,
            feature_tuples=sorted(shown_pairs),
            base_url=np_base_url,
        ),
        generate_missing=GENERATE_MISSING_LOCAL_EXPLANATIONS,
        local_db_url=np_cfg.get("local_db_url"),
    )
    explanation_note = f"read from the local Neuronpedia, {len(coverage.generated_artifacts)} generated this run" + (
        f", {len(coverage.generation_failures)} failed to generate" if coverage.generation_failures else ""
    )
explanations = resolve_feature_explanations(
    model_id=spec.neuronpedia_model,
    source_set=spec.neuronpedia_set,
    feature_tuples=sorted(shown_pairs),
    base_url=np_base_url,
)

feature_rows = []
for name, feats in shown.items():
    for rank, ((layer, pos, index), score) in enumerate(feats, start=1):
        feature_rows.append(
            (
                name if rank == 1 else "",
                Signed(graphs[name]["gap_delta"]) if rank == 1 else "",
                rank,
                layer,
                f"{pos} {tokenizer.decode([prompt_ids[pos]])!r}",
                index,
                Signed(score, "+.3g"),
                explanations.get((layer, index), ""),
                f"blocks.{layer}.{tc_hooks['feature_input_hook']}",
                f"blocks.{layer}.{tc_hooks['feature_output_hook']}",
            )
        )
display_notebook_table(
    feature_rows,
    [
        "stage",
        "gap change",
        "rank",
        "layer",
        "position",
        "feature",
        "influence",
        "explanation",
        "encoder reads",
        "decoder writes",
    ],
    title=f"Top {TOP_FEATURES_SHOWN} features of each stage's attribution graph ({spec.neuronpedia_set})",
    caption=f"Gap change: intervening on the stage's selected features. Explanations {explanation_note}; "
    f"{len(explanations)} of {len(shown_pairs)} features have one.",
    legend=SIGN_LEGEND,
)

## 6. Projections, and how far apart the groups really are

Two UMAP projections of the per-example states, and one of the stage directions. UMAP keeps neighbourhoods, not
distances: which points sit together is meaningful, while the size of a gap between clusters, and the axes, are not.

- **All four states.** Every probe example's answer, context, projected and final states in one plane, coloured by state
  and marked by group. The answer states sit far from the rest, which squeezes the other states together.
- **Without the answer states.** The same examples' context, projected and final states fitted on their own, so their
  arrangement is not dictated by the answer states.
- **The stage directions**, which agree when they sit together.

The separation table measures what the projections only suggest, in the full residual space: for each state, the
distance between the two groups' centroids against the spread of examples around them (a ratio above 1 means the groups
are further apart than they are wide), the cosine between the centroids, and leave-one-out nearest-centroid accuracy.
With two to four examples a group, treat it as a description of these examples, not an estimate.

In [ ]:
# @title 6: Trajectory and direction projections { display-mode: "form" }
import pandas as pd

trajectory = report["projections"]["trajectory"]
fig = px.scatter(
    pd.DataFrame(trajectory["records"]),
    x="x",
    y="y",
    color="state",
    symbol="group",
    hover_data=["example_index"],
    title=f"Per-example latent states ({trajectory['method']}, {trajectory['backend']})",
)
fig.update_layout(height=550, width=850)
display_plotly_figure(fig, "it-trajectory-projection")

trimmed = report["projections"]["trajectory_without_answer"]
fig = px.scatter(
    pd.DataFrame(trimmed["records"]),
    x="x",
    y="y",
    color="state",
    symbol="group",
    hover_data=["example_index"],
    title=f"Per-example latent states without the answer states ({trimmed['method']}, {trimmed['backend']})",
)
fig.update_layout(height=550, width=850)
display_plotly_figure(fig, "it-trajectory-without-answer")

display_notebook_table(
    [
        (
            row["state"],
            f"{row['centroid_distance']:.1f}",
            f"{row['within_group_spread']:.1f}",
            f"{row['separation_ratio']:.2f}",
            f"{row['centroid_cosine']:.3f}",
            f"{row['leave_one_out_accuracy']:.2f}",
        )
        for row in report["state_separation"]
    ],
    ["state", "centroid distance", "within-group spread", "ratio", "centroid cosine", "leave-one-out accuracy"],
    title="State separation between the groups, in the full residual space",
    caption=f"{len(examples)} store examples; unweighted centroids.",
)

directions = report["projections"]["directions"]
fig = px.scatter(
    pd.DataFrame(directions["records"]),
    x="x",
    y="y",
    color="basis",
    symbol="mode",
    text="stage",
    title=f"Stage directions ({directions['method']}, {directions['backend']})",
)
fig.update_traces(textposition="top center")
fig.update_layout(height=550, width=850)
display_plotly_figure(fig, "it-direction-projection")

## What to look for

What the tables and figures support, for this model, this prompt and these six store examples:

- **Three families, not one direction.** The embedding, answer-state and context-state stages form separate blocks in
  the cosine matrix: the same concept is carried along nearly unrelated directions at different points in the
  computation.
- **The answer states separate the groups most, and part of that is the token being read.** The answer state is read at
  the appended answer token, so the two groups' answer states differ in their input token (`Fruit` or `Color`) as well
  as in what the model computed. In the separation table their ratio is several times the context states'. Read the
  answer-state stages as "where the answer is", not as an earlier representation of the concept.
- **The context states separate the groups, on top of a large shared part.** Read at the probe word, every example sits
  nearer its own group's centroid, but the centroids' cosine is high: most of each state is common to both senses, and
  the concept is a small difference on top of it. That is why the context-state directions are small against the state
  norms, and why their projection looks mixed.
- **The projected and final states coincide here.** Every probe word is found in its prompt, so the final state is the
  projected state; overlapping points in the projections are that identity, not a finding.
- **Readout is not causation.** The final-site column ranks the embedding stages far above the store stages, because
  they are built from (or close to) the target unembeddings. A near-zero readout says a direction is nearly orthogonal
  to the target unembeddings, not that it cannot steer: at block 10 the store stages produce effects of the same order
  as the embedding stages once each is scaled by its own separation.
- **Past one separation, the families part.** At twice their separation the context-state stages flip the gap, while
  the answer-state stages' effect collapses: a push that large leaves the distribution those states came from.
- **Shared features, opposite effects.** The answer-state and context-state graphs share their mid-layer features at
  the answer position and differ in their late features, which is where the sign of their effect is decided.
- **Read a selected feature by what it writes.** Attribution ranks features by their decoder's effect on the gap, so a
  feature's explanation (what it fires on) can name something unrelated to fruit or color; the hook columns say where
  each one reads and writes.

What not to conclude:

- **From UMAP, only neighbourhoods.** The gap between two clusters, a cluster's size and the axes carry no meaning; use
  the separation table for distances. A projection fitted with the answer states is shaped by them, which is why the
  second projection leaves them out.
- **Beyond these examples.** Four fruit and two color examples, one prompt, one model: the numbers describe this
  configuration, and two of the color probes are not in the store at all, because the model answers them `Fruit`.
- **That a stage is "the" concept direction.** Each family steers, by different means and at different sites; which
  one to use depends on whether the question is about the answer, the word in context, or the unembedding.

The measured numbers for this configuration are discussed on
[issue #971](https://github.com/speediedan/interpretune/issues/971).